# Phase 5g — Lyrics Chunking Sensitivity (NEXT_STEPS step 8b)

Phase 3 passed each lyric to `all-mpnet-base-v2` in one piece, and sentence-transformers silently truncates the input at
`max_seq_length` = 384 tokens. Phase 3b also embedded every lyric in chunks (greedy line packing into ≤ 382-token
chunks, token-weighted mean of the L2-normalised chunk embeddings; `text_sbert_mpnet_chunk.npy`). Phase 5e compared
the two for lyrics-only retrieval and probes only. The main system (5f) and MuQ coherence (4c) still use the
truncated vector. This notebook checks whether that choice matters for the paper's results.

1. **How much text is lost:** token counts per lyric, share of lyrics truncated and share of tokens dropped.
2. **Retrieval:** the Phase 5f protocol (shared splits, exact cosine top-K, raw/z and α chosen on validation) with
   truncated vs chunked SBERT, for lyrics-only, MuQ + SBERT late fusion and early fusion. Paired tests on all test
   queries and on the truncated-lyric queries only (where any effect must show up).
3. **Coherence:** the Phase 4c protocol (5-fold cross-fitting, 200 bootstrap probes, same seed) with the chunked
   lyrics probe. The truncated run is recomputed and asserted equal to `coherence_muq.parquet` first.

New outputs: `lyrics_chunking_retrieval.csv`, `lyrics_chunking_tests.csv`, `coherence_muq_sbert_chunk.parquet`.
No existing file is modified. AV values and quadrants are evaluation targets only; `moods`, `genres`, `themes` are not read.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score
from transformers import AutoTokenizer

warnings.filterwarnings('ignore')
ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
SIGMA, KMAX, N_BOOT, SEED = 0.10, 20, 10_000, 42
ALPHAS  = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
SCHEMES = {'tvt_70_15_15': '70/15/15', 'tvt_40_30_30': '40/30/30'}
rng = np.random.default_rng(SEED)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
cat_sids  = lyrics_df['song_id'].tolist()
cat_pos   = {s: i for i, s in enumerate(cat_sids)}
assert pd.read_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'))['song_id'].tolist() == cat_sids
a_ids = pd.read_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'))['song_id'].tolist()
a_row = [{s: i for i, s in enumerate(a_ids)}[s] for s in cat_sids]

muq = np.load(os.path.join(ENC_DIR, 'audio_muq.npy'), mmap_mode='r')[a_row]
REPS = {'sbert_trunc': np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy')).astype(np.float64),
        'sbert_chunk': np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_chunk.npy')).astype(np.float64),
        'muq_L6': muq[:, 6].astype(np.float64), 'muq_L7': muq[:, 7].astype(np.float64)}
del muq

lab  = merged.set_index('song_id').loc[cat_sids]
quad = lab['quadrant'].values
Y    = lab[['arousal', 'valence']].values.astype(np.float64)
D_AV = np.linalg.norm(Y[:, None] - Y[None], axis=-1)
texts = lab['lyrics_text'].tolist()

mm = merged[merged['is_multimodal']]
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm['song_id'], mm['lyrics_path'])}
def read_split(modality, scheme, subset):
    folder = 'MERGE_Audio_Balanced' if modality == 'audio' else 'MERGE_Lyrics_Balanced'
    ids = pd.read_csv(os.path.join(ROOT, folder, 'tvt_dataframes', scheme,
                                   f'{scheme}_{subset}_{modality}_balanced.csv'))['Song']
    return set(ids.map(lyr_stem_to_sid).dropna()) if modality == 'lyrics' else set(ids)
cat = set(cat_sids)
shared = {lbl: {sub: sorted(read_split('audio', sc, sub) & read_split('lyrics', sc, sub) & cat)
                for sub in ['train', 'validate', 'test']} for sc, lbl in SCHEMES.items()}
print({lbl: {k: len(v) for k, v in d.items()} for lbl, d in shared.items()})

{'70/15/15': {'train': 1187, 'validate': 305, 'test': 302}, '40/30/30': {'train': 475, 'validate': 616, 'test': 617}}


## 1. How much lyric text does truncation drop?
Token counts use the `all-mpnet-base-v2` tokenizer without special tokens; the model sees at most 382 content tokens
(384 minus `<s>` and `</s>`).

In [2]:
tok = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
BUDGET = 384 - 2
n_tok = np.array([len(tok(t, add_special_tokens=False).input_ids) for t in texts])
trunc = n_tok > BUDGET
kept = np.minimum(n_tok, BUDGET)
print(f'Tokens per lyric: median {np.median(n_tok):.0f}, 90th pct {np.percentile(n_tok, 90):.0f}, max {n_tok.max()}')
print(f'Lyrics longer than {BUDGET} tokens (truncated in Phase 3): {trunc.sum()} / {len(trunc)} ({trunc.mean():.1%})')
print(f'Share of all lyric tokens dropped: {1 - kept.sum() / n_tok.sum():.1%}; '
      f'among truncated lyrics, median share dropped {np.median(1 - kept[trunc] / n_tok[trunc]):.1%}')
print(pd.crosstab(quad, trunc, normalize='index').round(3).rename(columns={False: 'fits', True: 'truncated'}).to_string())

l2 = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True)
cs = (l2(REPS['sbert_trunc']) * l2(REPS['sbert_chunk'])).sum(1)
print(f'\ncos(truncated, chunked): untruncated lyrics min {cs[~trunc].min():.4f}; '
      f'truncated lyrics median {np.median(cs[trunc]):.3f}, 5th pct {np.percentile(cs[trunc], 5):.3f}, min {cs[trunc].min():.3f}')
TRUNC_SIDS = set(np.array(cat_sids)[trunc])

Token indices sequence length is longer than the specified maximum sequence length for this model (718 > 512). Running this sequence through the model will result in indexing errors


Tokens per lyric: median 243, 90th pct 648, max 1485
Lyrics longer than 382 tokens (truncated in Phase 3): 471 / 2075 (22.7%)
Share of all lyric tokens dropped: 20.5%; among truncated lyrics, median share dropped 36.2%
col_0   fits  truncated
row_0                  
Q1     0.599      0.401
Q2     0.686      0.314
Q3     0.935      0.065
Q4     0.895      0.105

cos(truncated, chunked): untruncated lyrics min 1.0000; truncated lyrics median 0.970, 5th pct 0.898, min 0.760


## 2. Retrieval: truncated vs chunked SBERT
Machinery copied from Phase 5f (same `unit`, `topk`, `evaluate`, validation selection). MuQ is fixed at the 5f choice
(L6, z-scored); the SBERT preprocessing and the late-fusion α are chosen on validation separately for each variant.

In [3]:
_unit = {}
def unit(rep, how):
    if (rep, how) not in _unit:
        X = REPS[rep]; X = StandardScaler().fit_transform(X) if how == 'z' else X
        _unit[(rep, how)] = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return _unit[(rep, how)]

def score_matrix(cfg, rows):
    if cfg[0] == 'single':
        X = unit(*cfg[1]); return X[rows] @ X.T
    A, T = unit(*cfg[1]), unit(*cfg[2])
    if cfg[0] == 'late':
        return cfg[3] * (A[rows] @ A.T) + (1 - cfg[3]) * (T[rows] @ T.T)
    E = np.hstack([A, T]) / np.sqrt(2)
    return E[rows] @ E.T

DISC = 1 / np.log2(np.arange(2, KMAX + 2))
def topk(S, k):
    top = np.argpartition(-S, k, 1)[:, :k]
    return np.take_along_axis(top, np.take_along_axis(S, top, 1).argsort(1)[:, ::-1], 1)

def evaluate(cfg, qs):
    rows = np.array([cat_pos[s] for s in qs])
    S = score_matrix(cfg, rows); S[np.arange(len(rows)), rows] = -np.inf
    PK = np.column_stack([(quad[topk(S, k)] == quad[rows][:, None]).mean(1) for k in range(1, KMAX + 1)])
    top = topk(S, 10)
    d = np.take_along_axis(D_AV[rows], top, 1)
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    g = np.exp(-d**2 / (2 * SIGMA**2)); gi = np.exp(-np.sort(Dq, 1)[:, :10]**2 / (2 * SIGMA**2))
    return {'P@K': PK, 'nDCG@10': (g * DISC[:10]).sum(1) / (gi * DISC[:10]).sum(1),
            'AVdist@10': d.mean(1), 'song_id': np.array(qs)}

MUQ = ('muq_L6', 'z')
CFG = {}
for lbl in SCHEMES.values():
    val = shared[lbl]['validate']
    CFG[lbl] = {}
    for v in ['trunc', 'chunk']:
        t = max([(f'sbert_{v}', h) for h in ['raw', 'z']],
                key=lambda c: evaluate(('single', c), val)['P@K'][:, 9].mean())
        a = max(ALPHAS, key=lambda al: evaluate(('late', MUQ, t, al), val)['P@K'][:, 9].mean())
        CFG[lbl][f'lyrics-only [{v}]'] = ('single', t)
        CFG[lbl][f'late fusion [{v}]'] = ('late', MUQ, t, a)
        CFG[lbl][f'early fusion [{v}]'] = ('early', MUQ, t)
    print(lbl, {k: v[1:] for k, v in CFG[lbl].items() if not k.startswith('early')})

RES = {lbl: {name: evaluate(cfg, shared[lbl]['test']) for name, cfg in CFG[lbl].items()} for lbl in SCHEMES.values()}

# truncated rows must reproduce the Phase 5f main table
ref = pd.read_csv(os.path.join(OUT_DIR, 'main_results_table.csv'))
for lbl in SCHEMES.values():
    r = ref[ref.split == lbl].set_index('strategy')
    late5f = [s for s in r.index if s.startswith('late fusion (')][0]
    assert CFG[lbl]['late fusion [trunc]'][3] == float(late5f.split('=')[1][:-1])
    for mine, theirs in [('lyrics-only [trunc]', 'lyrics-only (SBERT)'), ('late fusion [trunc]', late5f),
                         ('early fusion [trunc]', 'early fusion (≡ α=0.5)')]:
        assert abs(RES[lbl][mine]['P@K'][:, 9].mean() - r.loc[theirs, 'P@10']) < 1e-9, (lbl, mine)
        assert abs(RES[lbl][mine]['nDCG@10'].mean() - r.loc[theirs, 'nDCG@10']) < 1e-9, (lbl, mine)
print('Truncated variant reproduces phase5f main_results_table.csv.')

70/15/15 {'lyrics-only [trunc]': (('sbert_trunc', 'z'),), 'late fusion [trunc]': (('muq_L6', 'z'), ('sbert_trunc', 'z'), 0.6), 'lyrics-only [chunk]': (('sbert_chunk', 'z'),), 'late fusion [chunk]': (('muq_L6', 'z'), ('sbert_chunk', 'z'), 0.6)}


40/30/30 {'lyrics-only [trunc]': (('sbert_trunc', 'z'),), 'late fusion [trunc]': (('muq_L6', 'z'), ('sbert_trunc', 'z'), 0.6), 'lyrics-only [chunk]': (('sbert_chunk', 'z'),), 'late fusion [chunk]': (('muq_L6', 'z'), ('sbert_chunk', 'z'), 0.6)}


Truncated variant reproduces phase5f main_results_table.csv.


In [4]:
def boot_ci(x, n=N_BOOT):
    idx = rng.integers(0, len(x), (n, len(x)))
    return np.percentile(x[idx].mean(1), [2.5, 97.5])

rows = []
for lbl, res in RES.items():
    for subset in ['all', 'truncated lyrics']:
        for name, m in res.items():
            sel = np.ones(len(m['song_id']), bool) if subset == 'all' else np.isin(m['song_id'], list(TRUNC_SIDS))
            r = {'split': lbl, 'queries': subset, 'strategy': name, 'cfg': str(CFG[lbl][name][1:]), 'n': int(sel.sum())}
            for col, x in [('P@10', m['P@K'][sel, 9]), ('nDCG@10', m['nDCG@10'][sel]), ('AVdist@10', m['AVdist@10'][sel])]:
                lo, hi = boot_ci(x); r.update({col: x.mean(), f'{col}_lo': lo, f'{col}_hi': hi})
            rows.append(r)
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT_DIR, 'lyrics_chunking_retrieval.csv'), index=False)
fmt = lambda r, c: f"{r[c]:.3f} [{r[c + '_lo']:.3f}, {r[c + '_hi']:.3f}]"
for (lbl, sub), s in table.groupby(['split', 'queries'], sort=False):
    print(f'\n=== {lbl} test, {sub} queries (n={s.n.iloc[0]}) ===')
    print(pd.DataFrame({c: s.apply(lambda r: fmt(r, c), axis=1).values for c in ['P@10', 'nDCG@10', 'AVdist@10']},
                       index=s.strategy.values).to_string())


=== 70/15/15 test, all queries (n=302) ===
                                      P@10               nDCG@10             AVdist@10
lyrics-only [trunc]   0.605 [0.568, 0.639]  0.427 [0.399, 0.455]  0.203 [0.191, 0.215]
late fusion [trunc]   0.702 [0.669, 0.733]  0.490 [0.461, 0.519]  0.178 [0.166, 0.190]
early fusion [trunc]  0.704 [0.673, 0.735]  0.488 [0.459, 0.517]  0.178 [0.166, 0.190]
lyrics-only [chunk]   0.601 [0.567, 0.637]  0.422 [0.394, 0.451]  0.203 [0.191, 0.216]
late fusion [chunk]   0.701 [0.669, 0.734]  0.486 [0.457, 0.515]  0.179 [0.167, 0.192]
early fusion [chunk]  0.699 [0.667, 0.731]  0.484 [0.456, 0.513]  0.180 [0.168, 0.192]

=== 70/15/15 test, truncated lyrics queries (n=72) ===
                                      P@10               nDCG@10             AVdist@10
lyrics-only [trunc]   0.674 [0.597, 0.747]  0.513 [0.451, 0.573]  0.181 [0.155, 0.209]
late fusion [trunc]   0.719 [0.651, 0.788]  0.546 [0.485, 0.605]  0.163 [0.137, 0.188]
early fusion [trunc]  0.721 [0

### Paired tests: chunked − truncated
Wilcoxon signed-rank (Pratt) on per-query scores with paired bootstrap 95% CIs, Holm-corrected within each
split × query subset × metric. The last row checks that the main fusion claim (late fusion beats lyrics-only)
holds with chunked lyrics.

In [5]:
def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

pairs = [('lyrics-only [chunk]', 'lyrics-only [trunc]'), ('late fusion [chunk]', 'late fusion [trunc]'),
         ('early fusion [chunk]', 'early fusion [trunc]'), ('late fusion [chunk]', 'lyrics-only [chunk]')]
trows = []
for lbl, res in RES.items():
    for subset in ['all', 'truncated lyrics']:
        sel = np.ones(len(res['lyrics-only [trunc]']['song_id']), bool) if subset == 'all' \
            else np.isin(res['lyrics-only [trunc]']['song_id'], list(TRUNC_SIDS))
        for metric, get in [('P@10', lambda m: m['P@K'][sel, 9]), ('nDCG@10', lambda m: m['nDCG@10'][sel])]:
            block = []
            for a, b in pairs:
                xa, xb = get(res[a]), get(res[b]); d = xa - xb; lo, hi = boot_ci(d)
                p = wilcoxon(xa, xb, zero_method='pratt').pvalue if (d != 0).any() else 1.0
                block.append({'split': lbl, 'queries': subset, 'n': int(sel.sum()), 'metric': metric,
                              'comparison': f'{a} − {b}', 'diff': d.mean(), 'lo': lo, 'hi': hi,
                              'share_queries_changed': (d != 0).mean(), 'p': p})
            for r, ph in zip(block, holm([r['p'] for r in block])):
                r['p_holm'] = ph
            trows += block
tests = pd.DataFrame(trows)
tests.to_csv(os.path.join(OUT_DIR, 'lyrics_chunking_tests.csv'), index=False)
print(tests.to_string(index=False, float_format=lambda x: f'{x:.3g}'))

   split          queries   n  metric                                  comparison      diff       lo        hi  share_queries_changed        p   p_holm
70/15/15              all 302    P@10   lyrics-only [chunk] − lyrics-only [trunc]  -0.00331  -0.0113   0.00464                  0.262    0.379    0.758
70/15/15              all 302    P@10   late fusion [chunk] − late fusion [trunc] -0.000662  -0.0053   0.00397                  0.132    0.741    0.758
70/15/15              all 302    P@10 early fusion [chunk] − early fusion [trunc]  -0.00497  -0.0106  0.000331                  0.139    0.117    0.351
70/15/15              all 302    P@10   late fusion [chunk] − lyrics-only [chunk]       0.1   0.0715     0.128                  0.738 1.09e-09 4.38e-09
70/15/15              all 302 nDCG@10   lyrics-only [chunk] − lyrics-only [trunc]  -0.00429 -0.00956   0.00106                  0.778    0.342    0.396
70/15/15              all 302 nDCG@10   late fusion [chunk] − late fusion [trunc]  -0.00

## 3. Coherence with the chunked lyrics probe
Phase 4c protocol: MuQ L7 audio probe, 5-fold cross-fitting stratified by quadrant, RidgeCV per fold, 200 bootstrap
probes per fold with the same seed. The bootstrap index draws are shared by the audio and lyrics probes within each
(fold, b), exactly as in 4c, so the audio side is identical for both lyric variants. The truncated run is checked
against `coherence_muq.parquet` before the chunked run is trusted.

In [6]:
N_BOOT_P, N_FOLDS, CENTER = 200, 5, np.array([0.5, 0.5])
RIDGE_ALPHAS = np.logspace(-2, 5, 15)
def make_probe(alpha=None):
    return make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS) if alpha is None else Ridge(alpha=alpha))
def cos_coh(A, L):
    A, L = A - CENTER, L - CENTER
    return (A * L).sum(-1) / (np.linalg.norm(A, axis=-1) * np.linalg.norm(L, axis=-1) + 1e-12)

X_a = REPS['muq_L7']
XL = {'trunc': np.vstack(lyrics_df['lyrics_sbert'].values).astype(np.float64), 'chunk': REPS['sbert_chunk']}
n = len(cat_sids)
rng_p = np.random.default_rng(SEED)
pred = {k: np.zeros((n, 2)) for k in ['audio', 'trunc', 'chunk']}
boot = {k: np.zeros((N_BOOT_P, n, 2)) for k in ['audio', 'trunc', 'chunk']}
for f, (itr, ite) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X_a, quad)):
    probes = {'audio': make_probe().fit(X_a[itr], Y[itr])}
    probes.update({v: make_probe().fit(XL[v][itr], Y[itr]) for v in XL})
    X = {'audio': X_a, **XL}
    for k, p in probes.items():
        pred[k][ite] = p.predict(X[k][ite])
    for b in range(N_BOOT_P):
        ib = rng_p.choice(itr, size=len(itr), replace=True)
        for k, p in probes.items():
            boot[k][b, ite] = make_probe(p[-1].alpha_).fit(X[k][ib], Y[ib]).predict(X[k][ite])
    print(f'fold {f}: ridge α ' + ', '.join(f'{k}={p[-1].alpha_:.0f}' for k, p in probes.items()))

for v in XL:
    print(f'OOF R² lyrics probe [{v}]: arousal {r2_score(Y[:, 0], pred[v][:, 0]):.3f}  valence {r2_score(Y[:, 1], pred[v][:, 1]):.3f}')

C = {}
for v in XL:
    lo, hi = np.percentile(cos_coh(boot['audio'], boot[v]), [2.5, 97.5], axis=0)
    C[v] = pd.DataFrame({'coh_cos': cos_coh(pred['audio'], pred[v]), 'lo': lo, 'hi': hi,
                         'status': np.select([hi < 0, lo > 0], ['robust paradox', 'robust coherent'], 'uncertain')},
                        index=cat_sids)

ref4c = pd.read_parquet(os.path.join(OUT_DIR, 'coherence_muq.parquet')).set_index('song_id').loc[cat_sids]
assert np.allclose(C['trunc'].coh_cos, ref4c.coh_cos) and np.allclose(C['trunc'].lo, ref4c.coh_cos_lo) \
    and np.allclose(C['trunc'].hi, ref4c.coh_cos_hi) and (C['trunc'].status == ref4c.status_cos).all()
print('Truncated run reproduces coherence_muq.parquet (point estimates, CIs and statuses).')

fold 0: ridge α audio=1000, trunc=3162, chunk=3162


fold 1: ridge α audio=1000, trunc=3162, chunk=3162


fold 2: ridge α audio=1000, trunc=3162, chunk=3162


fold 3: ridge α audio=1000, trunc=3162, chunk=3162


fold 4: ridge α audio=1000, trunc=3162, chunk=3162
OOF R² lyrics probe [trunc]: arousal 0.334  valence 0.518
OOF R² lyrics probe [chunk]: arousal 0.334  valence 0.520
Truncated run reproduces coherence_muq.parquet (point estimates, CIs and statuses).


In [7]:
t, c = C['trunc'], C['chunk']
print(f"Spearman(coherence trunc, chunk) = {spearmanr(t.coh_cos, c.coh_cos)[0]:.3f}; "
      f"max |Δ| on untruncated lyrics {np.abs(t.coh_cos - c.coh_cos)[~trunc].max():.3f}, on truncated {np.abs(t.coh_cos - c.coh_cos)[trunc].max():.3f}")
print(pd.crosstab(t.status.values, c.status.values, margins=True, rownames=['SBERT truncated'], colnames=['SBERT chunked']).to_string())
rp_t, rp_c = t.status == 'robust paradox', c.status == 'robust paradox'
print(f'\nRobust paradoxes: truncated {rp_t.sum()}, chunked {rp_c.sum()}, both {(rp_t & rp_c).sum()}')
print(f'Status changes among truncated lyrics: {(t.status != c.status)[trunc].sum()} / {trunc.sum()}; '
      f'among untruncated: {(t.status != c.status)[~trunc].sum()} / {(~trunc).sum()}')
gal = pd.read_csv(os.path.join(OUT_DIR, 'paradox_gallery_v2.csv'))
gid = gal['song_id'].tolist()
print(f'\nParadox gallery v2 ({len(gid)} tracks): statuses with chunked lyrics {c.loc[gid, "status"].value_counts().to_dict()}; '
      f'{sum(s in TRUNC_SIDS for s in gid)} of them have truncated lyrics')
both_enc = pd.read_csv(os.path.join(OUT_DIR, 'coherence_paradox_both_encoders.csv'))['song_id']
print(f'CLAP∩MuQ robust paradoxes ({len(both_enc)}): statuses with chunked lyrics {c.loc[both_enc, "status"].value_counts().to_dict()}')

# Quadrant pattern
qt = pd.DataFrame({v: pd.Series(C[v].status.values == 'robust paradox', index=cat_sids).groupby(quad).mean() for v in XL})
qt['mean_coh_trunc'] = pd.Series(t.coh_cos.values).groupby(quad).mean().values
qt['mean_coh_chunk'] = pd.Series(c.coh_cos.values).groupby(quad).mean().values
print('\nRobust paradox rate and mean coherence by quadrant:'); print(qt.round(3).to_string())

# Coherence vs retrieval quality (RQ3), 70/15/15 test queries, chunked lyrics for both coherence and retrieval
res = RES['70/15/15']; q = res['lyrics-only [chunk]']['song_id']
for name in ['lyrics-only [chunk]', 'late fusion [chunk]']:
    print(f"Spearman(chunked coherence, P@10 of {name}) = {spearmanr(c.loc[q, 'coh_cos'], res[name]['P@K'][:, 9])[0]:.3f}")
print(f"(truncated, as in 5f: lyrics {spearmanr(t.loc[q, 'coh_cos'], res['lyrics-only [trunc]']['P@K'][:, 9])[0]:.3f}, "
      f"fusion {spearmanr(t.loc[q, 'coh_cos'], res['late fusion [trunc]']['P@K'][:, 9])[0]:.3f})")

out = c.rename(columns={'coh_cos': 'coh_cos_chunk', 'lo': 'coh_cos_chunk_lo', 'hi': 'coh_cos_chunk_hi', 'status': 'status_cos_chunk'})
out['lyrics_truncated'] = trunc; out['n_tokens'] = n_tok
out.rename_axis('song_id').reset_index().to_parquet(os.path.join(OUT_DIR, 'coherence_muq_sbert_chunk.parquet'), index=False)
print('Saved: coherence_muq_sbert_chunk.parquet')

Spearman(coherence trunc, chunk) = 0.973; max |Δ| on untruncated lyrics 1.900, on truncated 1.906


SBERT chunked    robust coherent  robust paradox  uncertain   All
SBERT truncated                                                  
robust coherent             1334               0         20  1354
robust paradox                 0              60          6    66
uncertain                     27               4        624   655
All                         1361              64        650  2075

Robust paradoxes: truncated 66, chunked 64, both 60
Status changes among truncated lyrics: 37 / 471; among untruncated: 20 / 1604

Paradox gallery v2 (15 tracks): statuses with chunked lyrics {'robust paradox': 14, 'uncertain': 1}; 4 of them have truncated lyrics
CLAP∩MuQ robust paradoxes (15): statuses with chunked lyrics {'robust paradox': 14, 'uncertain': 1}

Robust paradox rate and mean coherence by quadrant:
    trunc  chunk  mean_coh_trunc  mean_coh_chunk
Q1  0.033  0.031           0.657           0.661
Q2  0.022  0.021           0.718           0.725
Q3  0.042  0.038           0.505       

## 4. Summary
See the printed results above; the findings are recorded under step 8b in `NEXT_STEPS.md`.